In [ ]:
# Import required libraries
import pandas as pd
from IPython.display import display, HTML
import sys
sys.path.append('src')
from parsers.parser_factory import ParserFactory
from utils.fraud_detector import FraudDetector

In [2]:
# Set pandas display options to show full data
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [3]:
# bank = 'hdfc'
# file_path = 'data/hdfc/Acct_Statement_XX0547_10052024.pdf'
# bank = 'icici'  # Change to 'icici' for ICICI statements
# file_path = 'data/icici/icici_statment.pdf'  # Change to your file path
bank = 'axis'
file_path = 'data/axis/Axis_sample4.pdf'

parser = ParserFactory.get_parser(bank)
transactions = parser.parse(file_path)
df = pd.DataFrame([t.__dict__ for t in transactions])
print(f"Parsed {len(transactions)} transactions.")

# Create a separate metadata DataFrame
metadata = parser.parse_metadata(file_path)
metadata_df = pd.DataFrame([metadata])
print("\nMetadata DataFrame:")
display(metadata_df)

# Filter transaction output columns (only if transactions were parsed)
if not df.empty:
    df = df[['date', 'narration', 'withdrawal_amt', 'deposit_amt', 'closing_balance']]
else:
    print("No transactions parsed — check file path or bank format.")

Parsed 46 transactions.

Metadata DataFrame:


,Bank,Account Holder,Account Number,Address,Joint Holder,Account Status,Branch,Email,Mobile,IFSC code,PAN,Customer ID,Scheme,A/C open date,From date,To date,Opening Balance,Closing Balance,Debit counts,Credit counts,Debit amount,Credit amount
0,Axis,TAVVA VENKATARAMAYYA,921010033551909,"FLAT NO 202 SRI KAILASH APP, PICKET WEST MAREDPALLY, SANTHOSHI MATA TEMPLE KAMAN,HYDERABAD,-, HYDERABAD",,,,VEXXXXAM@GMAIL.COM,XXXXXX2442,UTIB0000881,AGQPT9162N,217004522,PRESTIGE SALARY ACCOUNT,,01-01-2024,24-04-2024,,,,,,


In [ ]:
# Display transactions as scrollable HTML table with no-wrap on date/amount columns
html = df.to_html(index=False)

css = """
<style>
  .bsa-table th, .bsa-table td { padding: 4px 12px; }
  .bsa-table td:nth-child(1),
  .bsa-table td:nth-child(3),
  .bsa-table td:nth-child(4),
  .bsa-table td:nth-child(5) { white-space: nowrap; }
</style>
"""
html = html.replace('<table ', '<table class="bsa-table" ')
display(HTML(f'{css}<div style="height:600px; overflow:auto;">{html}</div>'))

date,narration,withdrawal_amt,deposit_amt,closing_balance
01-01-2024,MBBPay/AxisBankCr/2170045225/010124,29000.00,,49231.83
02-01-2024,IMPS/P2A/400221003620/TAVVAVENKATAR AMAYYA/X589831/KOTAKMAHINDRABAN,49100.00,,131.83
23-01-2024,IMPS/P2A/402315148146/CASHFREE/NSDLPA YM/BAV/9170933224429772001,,1.01,132.84
23-01-2024,IMPS/P2A/402317915223/VRINDAFI/ICICIBAN /IMPSXICI/9191125391129229216,,35280.00,35412.84
23-01-2024,IMPS/P2A/402317796619/TAVVAVENKATAR AMAYYA/X589831/KOTAKMAHINDRABAN,35400.00,,12.84
31-01-2024,Salary for Jan 2025,,88311.92,88324.76
01-02-2024,IMPS/P2A/403210493697/TAVVAVENKATAR AMAYYA/X589831/KOTAKMAHINDRABAN,88300.00,,24.76
03-02-2024,MOB/TPFT/GADA SAMPATH KU/916010005829403,,40000.00,40024.76
06-02-2024,IMPS/P2A/403721337549/TAVVAVENKATAR AMAYYA/X589831/KOTAKMAHINDRABAN,39900.00,,124.76
12-02-2024,IMPS/P2A/404322942628/RZPXPRIV/ICICIBA N/RKBansal/9158911232749229091,,1.00,125.76


In [ ]:
# Fraud Detection Analysis
detector = FraudDetector()
report = detector.analyze(transactions, metadata, file_path=file_path)

score = report['risk_score']
level = report['risk_level']
flags = report['flags']
summary = report['summary']

# Pick colour based on risk level
colour_map = {'Low': '#2ecc71', 'Moderate': '#f39c12', 'High': '#e67e22', 'Very High': '#e74c3c'}
colour = colour_map.get(level, '#95a5a6')

score_html = f"""
<div style="font-family:sans-serif; margin-bottom:16px;">
  <span style="font-size:18px; font-weight:bold;">Fraud Risk Score: </span>
  <span style="font-size:22px; font-weight:bold; color:{colour};">{score}/100 — {level} Risk</span>
  <span style="margin-left:24px; color:#555;">
    &nbsp;Flags: {summary['total_flags']} total
    &nbsp;|&nbsp; <span style="color:#e74c3c;">HIGH: {summary['high']}</span>
    &nbsp;|&nbsp; <span style="color:#f39c12;">MEDIUM: {summary['medium']}</span>
    &nbsp;|&nbsp; <span style="color:#3498db;">LOW: {summary['low']}</span>
  </span>
</div>
"""

if flags:
    rows = ""
    sev_colours = {'HIGH': '#e74c3c', 'MEDIUM': '#f39c12', 'LOW': '#3498db'}
    for f in flags:
        sc = sev_colours.get(f['severity'], '#95a5a6')
        rows += f"""
        <tr>
          <td style="white-space:nowrap; color:{sc}; font-weight:bold; padding:4px 12px;">{f['severity']}</td>
          <td style="white-space:nowrap; padding:4px 12px;">{f['check']}</td>
          <td style="padding:4px 12px;">{f['detail']}</td>
        </tr>"""
    flags_html = f"""
    <table style="border-collapse:collapse; width:100%; font-family:sans-serif; font-size:13px;">
      <thead>
        <tr style="background:#f0f0f0;">
          <th style="padding:6px 12px; text-align:left;">Severity</th>
          <th style="padding:6px 12px; text-align:left;">Check</th>
          <th style="padding:6px 12px; text-align:left;">Detail</th>
        </tr>
      </thead>
      <tbody>{rows}</tbody>
    </table>"""
else:
    flags_html = '<p style="font-family:sans-serif; color:#2ecc71;">✓ No fraud indicators detected.</p>'

display(HTML(score_html + flags_html))